# 04 — Iterative prediction & export

Loads the feature tables from notebook 03, runs iterative LGB predictions for all
train/test + inference_only locations, computes VVR crossing years, and exports a
full GeoPackage.

**Inputs** (`03_features/EXPERIMENT/`):
- `region_features.parquet` — 7,444 train/test regions
- `region_inference_features.parquet` — 940 inference_only regions

**Outputs** (`04_model_outputs/EXPERIMENT/`):
- `wocu_lgb_predictions_EXPERIMENT.gpkg` — full prediction GeoPackage

**Acceptance criteria** (section 6):
1. `region_features` + `region_inference_features` cover expected location counts
2. `predicted_bank_positions` has `n_train_test + n_inference_only` unique location IDs
3. Output GPKG contains `predicted_bank_positions`, `vvr_rates_of_change`, `summary_scope`, `all_lines`
4. `is_nvo` column is present as integer (0/1), no pyogrio warnings
5. `signaleringslijn` layer is present in output GPKG

In [ ]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (candidate / 'src').exists():
        _backend = candidate
        break
else:
    _backend = _cwd

os.chdir(_backend)
sys.path.insert(0, str(_backend))
print('cwd:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
from pyogrio import list_layers

import src.paths as PATHS
import src.model.baseline_model as BM
from src.model.export_utils import load_model_bundle
from src.model.feature_shifter import FeatureShiftConfig
from src.model.predictor import predict_iterative_baseline, predict_iterative_ml
from src.model.model_loader import ModelLoader
from src.erosion.centerline_utils import (
    build_ref_geom_lookup,
    compute_vvr_crossing_year,
    ensure_location_id_column,
    get_nvo_location_ids,
    point_from_offset,
)
from src.erosion.export import export_predictions

DATA_DIR = PATHS.DATA_DIR

print('Imports OK')

In [ ]:
# ── Experiment config ─────────────────────────────────────────────────────────
EXPERIMENT   = '20260314'
START_YEAR   = 2026
END_YEAR     = 2050
STEP         = 1
N_POINTS     = 3

FEATURES_DIR  = DATA_DIR / f'03_features/{EXPERIMENT}'
MODEL_OUT_DIR = DATA_DIR / f'04_model_outputs/{EXPERIMENT}'
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Source data
POSTPROC_GPKG     = DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260310.gpkg'
RAW_GPKG          = DATA_DIR / '01_raw/erosion/wocu_output_fase2_20260210.gpkg'
SIGNALERING_GPKG  = DATA_DIR / '01_raw/scope/20260205_signaleringslijn.gpkg'
SIGNALERING_LAYER = 'Vlak_vrije_ruimte_natuurvriendelijke_oever_ln'

# Model paths
BUNDLE_DIR    = DATA_DIR / '04_model_outputs/20260312'

PREDICTION_YEARS = list(range(START_YEAR, END_YEAR + 1, STEP))
print(f'Experiment       : {EXPERIMENT}')
print(f'Prediction years : {PREDICTION_YEARS[0]}–{PREDICTION_YEARS[-1]}  ({len(PREDICTION_YEARS)} years)')
print(f'Model output     : {MODEL_OUT_DIR}')

## 1. Load feature tables and build start_points

In [ ]:
features_df  = pd.read_parquet(FEATURES_DIR / 'region_features.parquet')
inference_df = pd.read_parquet(FEATURES_DIR / 'region_inference_features.parquet')
region_split = pd.read_parquet(FEATURES_DIR / 'region_split.parquet')

print(f'region_features         : {len(features_df):,} locations')
print(f'region_inference_features: {len(inference_df):,} locations')
print(f'Columns: {list(features_df.columns)}')

In [ ]:
# Build unified starting-point lookup: {loc_id: {last_dist, last_year, v_hist}}
# Priority: features_df (dist_t3, t3 year, v_train), then inference_df (dist_t2, t2, v_train)
start_points = {}

for loc_id, row in features_df.iterrows():
    t3_year = int(region_split.loc[loc_id, 't3']) if loc_id in region_split.index else int(row.get('t3', 2025))
    start_points[loc_id] = {
        'last_dist': row['dist_t3'],
        'last_year': t3_year,
        'v_hist':    row['v_train'],
    }

for loc_id, row in inference_df.iterrows():
    if loc_id not in start_points:
        start_points[loc_id] = {
            'last_dist': row['dist_t2'],
            'last_year': int(row['t2']),
            'v_hist':    row['v_train'],
        }

print(f'Total start_points: {len(start_points):,}')
last_years = pd.Series({k: v['last_year'] for k, v in start_points.items()})
print('last_year distribution:', last_years.value_counts().sort_index().to_dict())

## 2. Load LGB model bundle

In [ ]:
bundle = load_model_bundle(BUNDLE_DIR)
print(f'Bundle loaded from: {BUNDLE_DIR}')
print(f'  target    : {bundle.get("target")}')
print(f'  features  : {bundle.get("features")}')

## 3. Run iterative LGB prediction (rolling=True)

In [ ]:
# Combine train/test and inference feature tables for the ML predictor
# inference_df has same column schema (minus v_test, split) — concat on shared cols
shared_feature_cols = [c for c in features_df.columns if c in inference_df.columns]
all_features = pd.concat([
    features_df[shared_feature_cols],
    inference_df[shared_feature_cols],
])
print(f'Combined feature table: {all_features.shape}')
print(f'  Locations: {all_features.index.nunique():,}')

print(f'\nRunning LGB rolling prediction ({START_YEAR}–{END_YEAR}) ...')
df_lgb_rolling = predict_iterative_ml(
    bundle=bundle,
    features_df=all_features,
    start_points=start_points,
    model_name='lgb',
    start_year=START_YEAR,
    end_year=END_YEAR,
    step=STEP,
    rolling=True,
)
print(f'Predictions: {len(df_lgb_rolling):,} rows  ({df_lgb_rolling["location_id"].nunique():,} locations)')
print(df_lgb_rolling.groupby('year')['predicted_dist_m'].describe().round(2))

## 4. Load geometric inputs and build bank positions

In [ ]:
print('Loading geometric inputs ...')
scope_raw    = ensure_location_id_column(gpd.read_file(RAW_GPKG,       layer='vlakken_scope'))
centerlines  = ensure_location_id_column(gpd.read_file(RAW_GPKG,       layer='centrelines'))
bank_points  = ensure_location_id_column(gpd.read_file(RAW_GPKG,       layer='punten_oever'))
vvr          = gpd.read_file(POSTPROC_GPKG, layer='vvr_rates_of_change')
scope        = ensure_location_id_column(gpd.read_file(POSTPROC_GPKG,  layer='summary_scope'))
signalering  = gpd.read_file(SIGNALERING_GPKG, layer=SIGNALERING_LAYER).to_crs(28992)

cl_lookup       = centerlines.set_index('location_id')['geometry'].to_dict()
ref_geom_lookup = build_ref_geom_lookup(bank_points, n_points=N_POINTS)
nvo_location_ids = get_nvo_location_ids(vvr, scope)

print(f'Centerlines     : {len(cl_lookup):,}')
print(f'NVO locations   : {len(nvo_location_ids):,}')
print(f'Signaleringslijn: {len(signalering):,} features')

In [ ]:
print('Building predicted_bank_positions ...')
records = []
for row in df_lgb_rolling.itertuples(index=False):
    cline = cl_lookup.get(row.location_id)
    ref   = ref_geom_lookup.get(row.location_id)
    point = (
        point_from_offset(cline, row.predicted_dist_m, ref)
        if cline is not None and ref is not None and len(ref) > 0
        else None
    )
    records.append({
        'location_id':       row.location_id,
        'year':              row.year,
        'predicted_dist_m':  row.predicted_dist_m,
        'velocity_m_per_yr': row.velocity_m_per_yr,
        'is_nvo':            int(row.location_id in nvo_location_ids),  # int for GPKG compat
        'geometry':          point,
    })

predicted_bank_positions = gpd.GeoDataFrame(
    records, geometry='geometry', crs=centerlines.crs
)
valid = predicted_bank_positions.geometry.notna().sum()
print(f'predicted_bank_positions: {len(predicted_bank_positions):,} rows  ({valid:,} with geometry)')
print(f'  is_nvo=1: {predicted_bank_positions["is_nvo"].sum():,} rows')

## 5. VVR crossing year

In [ ]:
vvr_crossing = compute_vvr_crossing_year(
    df_lgb_rolling,
    nvo_location_ids,
    centerlines,
    scope_raw,
    signalering,
    reference_year=START_YEAR - 1,
)
crossed = vvr_crossing[vvr_crossing['crossing_year'].notna()]
print(f'VVR crossing: {len(vvr_crossing):,} NVO regions evaluated')
print(f'  Crossings within {START_YEAR}–{END_YEAR}: {len(crossed):,}')
print(f'  crossing_year distribution:')
print(crossed['crossing_year'].value_counts().sort_index().to_string())

## 6. Export GeoPackage

In [ ]:
OUTPUT_GPKG = MODEL_OUT_DIR / f'wocu_lgb_predictions_{EXPERIMENT}.gpkg'

output_path = export_predictions(
    base_gpkg=POSTPROC_GPKG,
    output_gpkg=OUTPUT_GPKG,
    predicted_bank_positions=predicted_bank_positions,
    vvr_crossing=vvr_crossing,
    scope_raw=scope_raw,
    signaleringslijn=signalering,
)
print(f'Exported → {output_path}  ({output_path.stat().st_size / 1e6:.1f} MB)')

## 7. Acceptance criteria

Run all checks. All must pass before the output GPKG is considered valid.

In [ ]:
import warnings as _w
_w.filterwarnings('error', category=UserWarning)  # catch pyogrio RuntimeWarnings

print('=== Acceptance criteria ===')
passed = []

# 1. Location counts
n_train_test   = len(features_df)
n_inference    = len(inference_df)
n_pred_unique  = predicted_bank_positions['location_id'].nunique()
expected_locs  = n_train_test + n_inference
c1 = n_pred_unique >= n_train_test  # inference_only may be partial if no geometry
passed.append(c1)
print(f'[{"OK" if c1 else "FAIL"}] 1. Unique prediction locations: {n_pred_unique:,} '
      f'(train/test={n_train_test:,} + inference={n_inference:,} → expected ~{expected_locs:,})')

# 2. Required GPKG layers present
_w.filterwarnings('ignore')  # suppress pyogrio layer warnings for this check
layers_in_gpkg = {name for name, _ in list_layers(output_path)}
required_layers = {'predicted_bank_positions', 'vvr_rates_of_change', 'summary_scope', 'all_lines'}
missing_layers = required_layers - layers_in_gpkg
c2 = len(missing_layers) == 0
passed.append(c2)
print(f'[{"OK" if c2 else "FAIL"}] 2. Required layers present: {sorted(layers_in_gpkg)}')
if missing_layers: print(f'         MISSING: {missing_layers}')

# 3. is_nvo column is integer, no missing
pbp = gpd.read_file(output_path, layer='predicted_bank_positions')
c3a = 'is_nvo' in pbp.columns
c3b = pbp['is_nvo'].dtype in [int, 'int64', 'int32'] if c3a else False
c3 = c3a and c3b
passed.append(c3)
print(f'[{"OK" if c3 else "FAIL"}] 3. is_nvo present and integer dtype: '
      f'present={c3a}, dtype={pbp["is_nvo"].dtype if c3a else "N/A"}')

# 4. signaleringslijn layer present
c4 = 'signaleringslijn' in layers_in_gpkg
passed.append(c4)
print(f'[{"OK" if c4 else "FAIL"}] 4. signaleringslijn layer in GPKG')

# 5. vvr_rates_of_change has crossing year column
vvr_out = gpd.read_file(output_path, layer='vvr_rates_of_change')
c5 = 'predicted_vvr_crossing_year' in vvr_out.columns
passed.append(c5)
n_filled = vvr_out['predicted_vvr_crossing_year'].notna().sum() if c5 else 0
print(f'[{"OK" if c5 else "FAIL"}] 5. predicted_vvr_crossing_year on vvr layer: {n_filled:,}/{len(vvr_out):,} filled')

# Summary
n_pass = sum(passed)
print(f'\n{n_pass}/{len(passed)} checks passed.')
if n_pass == len(passed):
    print('✓ ALL ACCEPTANCE CRITERIA MET — GPKG is valid.')
else:
    print('✗ Some checks failed — review before sharing the GPKG.')

## 8. Layer inventory

In [ ]:
import warnings
warnings.filterwarnings('ignore')
print(f'=== Layer inventory: {output_path.name} ===')
for name, geom in list_layers(output_path):
    gdf = gpd.read_file(output_path, layer=name)
    pred_cols = [c for c in gdf.columns if 'predict' in c or 'crossing' in c]
    extra = f'  cols: {pred_cols}' if pred_cols else ''
    print(f'  {name:45s} [{geom or "None":12s}]  {len(gdf):>7,} rows{extra}')